This code serves as the merger/cleaning code for the master dataset for our group. It combines Census data (2 files), WARN Layoff Data, and ZHVI home value data and produces one csv.

After all libraries and csvs are loaded, each sheet is individually cleaned to ensure it is in the proper formatting for analysis. The Zillow data is cleaned first, then WARN data, then both Census sheets. Once done, all are merged on the state and year to create one master table, which is then exported as a csv. This csv is what each group member uses and adds onto for their own individual analysis. 

In [1]:
library(dplyr)
library(tidyr)
library(lubridate)
library(readr)
library(stringr)

Warn <- read.csv("warn-excluding-2026.csv", check.names = FALSE)
Zhvi <- read.csv("zhvi-allhomes-smoothed-seasonal.csv", check.names = FALSE)
Census <- read.csv("census-data-2025.csv")
Oldcensus <- read.csv("census-data-2020.csv")


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union




In [2]:
zhvi_states <- Zhvi[Zhvi$RegionType == "state", ]
date_cols <- grep("^\\d{4}-\\d{2}-\\d{2}$", names(zhvi_states), value = TRUE)
zhvi_long <- zhvi_states %>% pivot_longer(cols = all_of(date_cols),names_to = "month",values_to = "zhvi" )
zhvi_long$state <- trimws(zhvi_long$RegionName)
zhvi_long$month <- as.Date(zhvi_long$month)
zhvi_long$year <- year(zhvi_long$month)
zhvi_long$zhvi <- as.numeric(zhvi_long$zhvi)
zhvi_annual <- zhvi_long %>% group_by(state, year) %>% summarise(zhvi_avg = mean(zhvi, na.rm = TRUE),.groups = "drop")

In [3]:
warn_clean <- Warn %>% 
  mutate(
    state = trimws(State),
    workers = parse_number(`Number of Workers`),
    eff_raw = str_extract(`Effective Date`, "\\d{1,2}/\\d{1,2}/\\d{2,4}"),
    rec_raw = str_extract(`WARN Received Date`, "\\d{1,2}/\\d{1,2}/\\d{2,4}")
  )

warn_clean$eff_date <- as.Date(parse_date_time(warn_clean$eff_raw, orders = c("mdy", "m/d/Y", "m/d/y"), quiet = TRUE))
warn_clean$rec_date <- as.Date(parse_date_time(warn_clean$rec_raw, orders = c("mdy", "m/d/Y", "m/d/y"), quiet = TRUE))
warn_clean$event_date <- coalesce(warn_clean$eff_date, warn_clean$rec_date)
warn_clean$closure_layoff <- tolower(coalesce(warn_clean$`Closure/Layoff`, ""))

warn_clean <- warn_clean %>% 
  filter(!is.na(state), !is.na(event_date)) %>% 
  mutate(
    year = year(event_date),
    is_closure = if_else(grepl("clos", closure_layoff), 1, 0),
    is_layoff = if_else(grepl("layoff", closure_layoff), 1, 0)
  )

warn_annual <- warn_clean %>% 
  group_by(state, year) %>% 
  summarise(
    warn_events = n(),
    workers_affected = sum(workers, na.rm = TRUE),
    closures = sum(is_closure, na.rm = TRUE),
    layoffs = sum(is_layoff, na.rm = TRUE),
    .groups = "drop"
  )

In [4]:
census_states <- Census %>%
  filter(SUMLEV == 40) %>%
  mutate(state = trimws(NAME))

census_2020 <- census_states %>% transmute(state, year = 2020, pop = POPESTIMATE2020, births = BIRTHS2020, deaths = DEATHS2020, netmig = NETMIG2020)
census_2021 <- census_states %>% transmute(state, year = 2021, pop = POPESTIMATE2021, births = BIRTHS2021, deaths = DEATHS2021, netmig = NETMIG2021)
census_2022 <- census_states %>% transmute(state, year = 2022, pop = POPESTIMATE2022, births = BIRTHS2022, deaths = DEATHS2022, netmig = NETMIG2022)
census_2023 <- census_states %>% transmute(state, year = 2023, pop = POPESTIMATE2023, births = BIRTHS2023, deaths = DEATHS2023, netmig = NETMIG2023)
census_2024 <- census_states %>% transmute(state, year = 2024, pop = POPESTIMATE2024, births = BIRTHS2024, deaths = DEATHS2024, netmig = NETMIG2024)
census_2025 <- census_states %>% transmute(state, year = 2025, pop = POPESTIMATE2025, births = BIRTHS2025, deaths = DEATHS2025, netmig = NETMIG2025)

census_annual <- bind_rows(
  census_2020, census_2021, census_2022, census_2023, census_2024, census_2025
)

In [5]:
oldcensus_states <- Oldcensus %>%
  filter(SUMLEV == 40) %>%
  mutate(state = trimws(NAME))

old_2010 <- oldcensus_states %>% transmute(state, year = 2010, pop = POPESTIMATE2010, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2011 <- oldcensus_states %>% transmute(state, year = 2011, pop = POPESTIMATE2011, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2012 <- oldcensus_states %>% transmute(state, year = 2012, pop = POPESTIMATE2012, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2013 <- oldcensus_states %>% transmute(state, year = 2013, pop = POPESTIMATE2013, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2014 <- oldcensus_states %>% transmute(state, year = 2014, pop = POPESTIMATE2014, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2015 <- oldcensus_states %>% transmute(state, year = 2015, pop = POPESTIMATE2015, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2016 <- oldcensus_states %>% transmute(state, year = 2016, pop = POPESTIMATE2016, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2017 <- oldcensus_states %>% transmute(state, year = 2017, pop = POPESTIMATE2017, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2018 <- oldcensus_states %>% transmute(state, year = 2018, pop = POPESTIMATE2018, births = NA_real_, deaths = NA_real_, netmig = NA_real_)
old_2019 <- oldcensus_states %>% transmute(state, year = 2019, pop = POPESTIMATE2019, births = NA_real_, deaths = NA_real_, netmig = NA_real_)

census_old <- bind_rows(old_2010, old_2011, old_2012, old_2013, old_2014, old_2015, old_2016, old_2017, old_2018, old_2019)

census_annual_all <- bind_rows(census_old, census_annual) %>%
  distinct(state, year, .keep_all = TRUE)

In [ ]:
master_annual <- left_join(zhvi_annual, warn_annual, by = c("state", "year"))
master_annual <- left_join(master_annual, census_annual_all, by = c("state", "year"))

master_annual <- master_annual %>%
  mutate(
    warn_events = if_else(is.na(warn_events), 0L, warn_events),
    workers_affected = if_else(is.na(workers_affected), 0, workers_affected),
    closures = if_else(is.na(closures), 0L, closures),
    layoffs = if_else(is.na(layoffs), 0L, layoffs)
  ) %>%
  filter(year <= 2025) %>%
  filter(!is.na(pop)) %>%
  arrange(state, year)

nrow(master_annual)
sum(duplicated(master_annual[c("state", "year")]))
print(master_annual, n = 30, width = Inf)

[1] 816

[1] 0

# A tibble: 816 × 11
   state    year zhvi_avg warn_events workers_affected closures layoffs     pop
   <chr>   <dbl>    <dbl>       <int>            <dbl>    <dbl>   <dbl>   <int>
 1 Alabama  2010  125684.          30             3993       21       9 4785514
 2 Alabama  2011  119875.          36             5445       22      14 4799642
 3 Alabama  2012  121659.          32             5156       18      14 4816632
 4 Alabama  2013  128972.          41             7329       26      15 4831586
 5 Alabama  2014  133206.          17             4454        9       8 4843737
 6 Alabama  2015  135095.          26             4946       13      13 4854803
 7 Alabama  2016  139963.          21             3566       10      11 4866824
 8 Alabama  2017  144021.          12             2200        5       7 4877989
 9 Alabama  2018  149763.          28             3474       19       9 4891628
10 Alabama  2019  158084.          17             1847       10       7 4907965
11 Alabama  2020  1

In [14]:
write_csv(master_annual,"master_clean.csv")